/data/2_data_server/cv-07/anaconda3/envs/beit3_new/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
import pandas as pd
import json
from transformers import XLMRobertaTokenizer
from collections import defaultdict
import os

# 1. 토크나이저 초기화 (제공해주신 경로 사용)
# XLMRobertaTokenizer는 SentencePiece 모델을 사용합니다.
# `spm_model_file` 매개변수를 사용하여 직접 경로를 지정할 수 있습니다.
tokenizer = XLMRobertaTokenizer(spm_model_file="/data/2_data_server/cv-07/dice/2025_samsung_challenge/unilm/beit3/beit3.spm")

# 2. VQA 데이터셋 CSV 파일 경로 (실제 파일 경로로 변경하세요)
vqa_csv_path = "/data/2_data_server/cv-07/dice/2025_samsung_challenge/data/data_2/test.csv" # 예: "vqa_test_dataset.csv"
output_jsonl_path = "coco_retrieval_format_for_beit3.jsonl"
image_base_dir = "./test_input_images/" # 이미지 파일들이 있는 기본 경로

# 3. CSV 데이터 로드
try:
    df = pd.read_csv(vqa_csv_path)
except FileNotFoundError:
    print(f"Error: CSV file not found at {vqa_csv_path}")
    print("Please update 'vqa_csv_path' to your actual VQA dataset CSV file.")
    exit()

# 이미지 경로에 고유 ID 부여를 위한 맵
image_path_to_id = defaultdict(lambda: len(image_path_to_id))

# 변환된 데이터를 저장할 리스트
converted_data = []

# 4. 데이터 변환 및 토큰화
for index, row in df.iterrows():
    img_relative_path = row["img_path"] # CSV에서 상대 경로로 가정
    
    # img_path에서 이미지 파일명만 추출하여 image_id 생성에 활용하거나, 
    # 전체 경로에 대해 고유 ID를 부여할 수 있습니다.
    # 여기서는 이미지 경로를 기반으로 고유 ID를 부여합니다.
    full_image_path = os.path.join(image_base_dir, img_relative_path.lstrip('./')) # './' 제거

    # 동일한 이미지에 대해 같은 image_id를 부여
    current_image_id = image_path_to_id[full_image_path]

    # VQA 태스크의 경우, 일반적으로 질문과 정답 보기를 결합하여 검색 텍스트로 만듭니다.
    # 혹은 모든 보기를 각각 질문과 결합하여 여러 텍스트-이미지 쌍을 생성할 수도 있습니다.
    # 여기서는 'Question'과 'A' (가장 일반적인 경우 정답이라고 가정)를 결합합니다.
    # 실제 VQA 데이터셋에 '정답' 컬럼이 있다면 그를 활용하는 것이 좋습니다.
    
    # 예시: 질문과 보기 A를 결합
    question = row["Question"]
    answer_A = row["A"]
    combined_text = f"{answer_A}" # 예시: "What types of fruits are visible in the image? Bananas and grapes placed in baskets"

    # 텍스트 토큰화
    # `add_special_tokens=False`를 사용하여 토크나이저가 자동으로 시작/끝 토큰을 추가하지 않도록 합니다.
    # BEiT-3 모델의 `RetrievalDataset`에서 처리할 가능성이 있습니다.
    # `tokenizer.encode`는 바로 ID 리스트를 반환합니다.
    token_ids = tokenizer.encode(combined_text, add_special_tokens=False)

    converted_data.append({
        "image_path": full_image_path, # 전체 경로 사용
        "text_segment": token_ids,
        "image_id": current_image_id
    })
    
    # 필요하다면 다른 보기(B, C, D)도 각각 결합하여 추가 항목을 생성할 수 있습니다.
    # 이렇게 하면 하나의 이미지에 대해 여러 텍스트 세그먼트를 매칭하게 됩니다.
    # 예를 들어,
    # token_ids_B = tokenizer.encode(f"{question} {row['B']}", add_special_tokens=False)
    # converted_data.append({"image_path": full_image_path, "text_segment": token_ids_B, "image_id": current_image_id})
    # ... 이런 식으로 모든 보기에 대해 추가 가능

# 5. JSON Lines 파일로 저장
print(f"Saving {len(converted_data)} entries to {output_jsonl_path}...")
with open(output_jsonl_path, 'w', encoding='utf-8') as f:
    for entry in converted_data:
        json.dump(entry, f, ensure_ascii=False)
        f.write('\n')
print("Conversion complete!")

# 6. RetrievalDataset.make_coco_dataset_index 호출
# 이 함수는 위에서 생성된 JSON Lines 파일을 기반으로 BEiT-3 모델이 학습/추론에 사용할 인덱스 파일을 생성합니다.
# `data_path`는 JSON Lines 파일이 있는 디렉토리가 되어야 합니다.
# 따라서, output_jsonl_path의 디렉토리를 `data_path`로 사용합니다.
# `file_names`는 생성된 JSON Lines 파일의 이름을 리스트로 전달합니다.
data_dir_for_retrieval = os.path.dirname(output_jsonl_path)
if not data_dir_for_retrieval: # 현재 디렉토리에 저장된 경우
    data_dir_for_retrieval = "."

# make_coco_dataset_index 함수가 이 파일을 직접 참조하므로, 파일명을 명시합니다.
# RetrievalDataset의 make_coco_dataset_index가 어떤 파일을 기대하는지 확인해야 합니다.
# 일반적으로는 디렉토리 내의 특정 파일명(예: "train.jsonl", "val.jsonl")을 기대합니다.
# 여기서는 예시로 output_jsonl_path의 파일명을 직접 전달하는 방식입니다.
# 실제 RetrievalDataset 구현에 따라 `make_coco_dataset_index`의 인자를 조정해야 할 수 있습니다.

# 아래는 RetrievalDataset.make_coco_dataset_index가 파일명 리스트를 인자로 받는 경우의 가상 코드입니다.
# 실제 `make_coco_dataset_index`의 내부 구현에 따라 인자 전달 방식이 달라질 수 있습니다.
# 예를 들어, 해당 함수가 특정 이름의 JSONL 파일을 기대한다면,
# output_jsonl_path를 해당 이름(예: "coco_retrieval/train.jsonl")으로 변경해야 합니다.

# 예시: RetrievalDataset 라이브러리에 따라 적절히 호출
# assuming `RetrievalDataset` expects the JSONL file to be within `data_path`
# and named specifically (e.g., train.jsonl or val.jsonl)
# You might need to rename `output_jsonl_path` to a recognized name.

# 임시로 파일을 해당 경로로 복사하거나, 함수가 해당 파일명을 받아들이도록 수정해야 할 수 있습니다.
# 아래는 가상의 시나리오로, `make_coco_dataset_index`가 해당 파일을 읽을 수 있다고 가정합니다.
# 실제 BEiT-3의 `RetrievalDataset` 코드를 확인하여 정확한 사용법을 따르세요.

# import RetrievalDataset (상단에 이미 import 되었다고 가정)
# from datasets import RetrievalDataset # 이미 import 된 줄입니다.

print(f"Calling RetrievalDataset.make_coco_dataset_index with data_path='{data_dir_for_retrieval}' and tokenizer.")
print("Please ensure that your `RetrievalDataset` implementation correctly reads the generated JSONL file.")
print("You might need to adjust the `output_jsonl_path` to match expected file names (e.g., 'train.jsonl', 'val.jsonl') within the `data_path`.")

# RetrievalDataset.make_coco_dataset_index(
#     data_path=data_dir_for_retrieval, # JSONL 파일이 있는 디렉토리
#     tokenizer=tokenizer,
#     # make_coco_dataset_index가 처리할 파일명을 지정하는 인자가 있다면 여기에 추가
#     # 예: file_names=["coco_retrieval_format_for_beit3.jsonl"] 
# )

print("\n---")
print("Important Note: The `RetrievalDataset.make_coco_dataset_index` function is likely to expect")
print("specific file names (e.g., 'train.jsonl', 'val.jsonl') within the `data_path` directory.")
print("You might need to rename or move the generated `coco_retrieval_format_for_beit3.jsonl`")
print("to a name and location that `make_coco_dataset_index` expects, or modify `make_coco_dataset_index` if you have access to its source.")

TypeError: __init__() missing 1 required positional argument: 'vocab_file'